Etsub Demile
DS4400
HW2

In [40]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Load data
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("Original data:")
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

#Divide price by 1000

train_df['price'] = train_df['price'] / 1000
test_df['price'] = test_df['price'] / 1000

print("\n Price divided by 1000")

Scale features to mean=0, std=1

# Get feature columns (exclude id, date, price)
feature_cols = [col for col in train_df.columns if col not in ['id', 'date', 'price']]

# Fit scaler on training data
scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])

print("Features scaled to mean=0, std=1")

#save

train_df.to_csv("train_preprocessed.csv", index=False)
test_df.to_csv("test_preprocessed.csv", index=False)

print("\n Preprocessed data saved:")
print("  - train_preprocessed.csv")
print("  - test_preprocessed.csv")

print("\nPreprocessed data summary:")
print(f"Feature means: {train_df[feature_cols].mean().mean():.6f}")
print(f"Feature stds: {train_df[feature_cols].std().mean():.6f}")
print(f"Price mean: {train_df['price'].mean():.2f}")
print(f"Price range: {train_df['price'].min():.2f} - {train_df['price'].max():.2f}")


Original data:
Train shape: (1000, 20)
Test shape: (1000, 22)

✓ Price divided by 1000
✓ Features scaled to mean=0, std=1

✓ Preprocessed data saved:
  - train_preprocessed.csv
  - test_preprocessed.csv

Preprocessed data summary:
Feature means: -0.000000
Feature stds: 1.000500
Price mean: 520.41
Price range: 80.00 - 3075.00


Problem 2

In [44]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

train_df = pd.read_csv("train_preprocessed.csv")
test_df = pd.read_csv("test_preprocessed.csv")

# Drop columns that aren't features
cols_to_drop_train = ['Unnamed: 0', 'price']
cols_to_drop_test = ['Unnamed: 0', 'id', 'date', 'price']

X_train = train_df.drop(columns=cols_to_drop_train)
y_train = train_df['price']
X_test = test_df.drop(columns=cols_to_drop_test)
y_test = test_df['price']

model = LinearRegression()
model.fit(X_train, y_train)

coefficients = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_
}).sort_values(by="Coefficient", key=abs, ascending=False)

print(coefficients)

y_train_pred = model.predict(X_train)
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

y_test_pred = model.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"\nTrain MSE: {train_mse:.4f}, Train R²: {train_r2:.4f}")
print(f"Test MSE: {test_mse:.4f}, Test R²: {test_r2:.4f}")

          Feature  Coefficient
8           grade    92.897547
14            lat    82.751754
11       yr_built   -72.824391
5      waterfront    63.678286
2     sqft_living    56.488927
6            view    49.623154
9      sqft_above    48.353975
16  sqft_living15    41.575151
10  sqft_basement    26.513263
13        zipcode   -24.317325
1       bathrooms    18.469939
12   yr_renovated    16.903636
0        bedrooms   -14.193346
7       condition    11.297767
17     sqft_lot15   -11.204930
4          floors    10.903800
3        sqft_lot    10.588014
15           long   -10.540199

Train MSE: 31119.8929, Train R²: 0.7297
Test MSE: 57161.5328, Test R²: 0.6572


The multiple linear regression model shows that grade (92.90), latitude (82.75), and waterfront (63.68) have the largest standardized coefficients, indicating that property quality and location are the most influential factors in predicting house prices. Since features are standardized, these coefficients represent the change in standardized price for a one standard deviation change in each feature, making them directly comparable.
The model fits the training data reasonably well, achieving an R² of 0.7297, meaning it explains about 73% of the variance in prices. On the test set, the R² drops to 0.6572, showing decent generalization with some reduction in performance. The training MSE is 31,119.89 while the testing MSE is 57,161.53 (both computed on standardized prices). The test error is approximately 1.8× the training error, suggesting some overfitting but not severe. Overall, the model provides a reasonably good fit, though there is room for improvement in generalization to unseen data.

Problem 3

In [45]:

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load preprocessed data
train_df = pd.read_csv("train_preprocessed.csv")
test_df = pd.read_csv("test_preprocessed.csv")

# Prepare features and target
cols_to_drop_train = ['Unnamed: 0', 'price']
cols_to_drop_test = ['Unnamed: 0', 'id', 'date', 'price']

X_train = train_df.drop(columns=cols_to_drop_train).values
y_train = train_df['price'].values.reshape(-1, 1)
X_test = test_df.drop(columns=cols_to_drop_test).values
y_test = test_df['price'].values.reshape(-1, 1)

# Implementation
def closed_form_linear_regression(X, y):
    """Closed-form solution: β = (X^T X)^(-1) X^T y"""
    X_b = np.c_[np.ones(X.shape[0]), X]  # add intercept
    beta = np.linalg.solve(X_b.T @ X_b, X_b.T @ y)
    return beta

def predict(X, beta):
    """Predict: y = X β"""
    X_b = np.c_[np.ones(X.shape[0]), X]
    return X_b @ beta

# Train model
beta_my = closed_form_linear_regression(X_train, y_train)
y_train_pred_my = predict(X_train, beta_my)
y_test_pred_my = predict(X_test, beta_my)

train_mse_my = mean_squared_error(y_train, y_train_pred_my)
train_r2_my = r2_score(y_train, y_train_pred_my)
test_mse_my = mean_squared_error(y_test, y_test_pred_my)
test_r2_my = r2_score(y_test, y_test_pred_my)

# Sklearn
model_sklearn = LinearRegression()
model_sklearn.fit(X_train, y_train)

y_train_pred_sklearn = model_sklearn.predict(X_train)
y_test_pred_sklearn = model_sklearn.predict(X_test)

train_mse_sklearn = mean_squared_error(y_train, y_train_pred_sklearn)
train_r2_sklearn = r2_score(y_train, y_train_pred_sklearn)
test_mse_sklearn = mean_squared_error(y_test, y_test_pred_sklearn)
test_r2_sklearn = r2_score(y_test, y_test_pred_sklearn)

# Comparison
results = pd.DataFrame({
    'Metric': ['Train MSE', 'Train R²', 'Test MSE', 'Test R²'],
    'My Implementation': [train_mse_my, train_r2_my, test_mse_my, test_r2_my],
    'Sklearn': [train_mse_sklearn, train_r2_sklearn, test_mse_sklearn, test_r2_sklearn],
    'Difference': [
        abs(train_mse_my - train_mse_sklearn),
        abs(train_r2_my - train_r2_sklearn),
        abs(test_mse_my - test_mse_sklearn),
        abs(test_r2_my - test_r2_sklearn)
    ]
})

print(results)

      Metric  My Implementation       Sklearn    Difference
0  Train MSE       31119.892884  31119.892884  3.637979e-12
1   Train R²           0.729715      0.729715  1.110223e-16
2   Test MSE       57161.532843  57161.532843  1.309672e-10
3    Test R²           0.657155      0.657155  8.881784e-16


The results demonstrate that my closed-form implementation produces virtually identical results to sklearn's LinearRegression package. The differences in metrics are negligible - on the order of 10⁻¹⁶ to 10⁻¹⁰, which are due to floating-point arithmetic precision rather than any meaningful algorithmic difference. Both implementations achieve a Train R² of 0.7297 and Test R² of 0.6572, with Train MSE of 31,119.89 and Test MSE of 57,161.53 (computed on standardized prices). This confirms that my implementation correctly uses the closed-form solution β = (XᵀX)⁻¹Xᵀy. The fact that sklearn's LinearRegression also uses the same closed-form solution internally explains why the results match exactly. The small difference between training and testing R² (0.730 vs 0.657) indicates reasonable generalization with no severe overfitting.

Problem 4

In [46]:

#extract sqft_living feature and price
X_train = train_df['sqft_living'].values.reshape(-1, 1)
y_train = train_df['price'].values.reshape(-1, 1)
X_test = test_df['sqft_living'].values.reshape(-1, 1)
y_test = test_df['price'].values.reshape(-1, 1)

def add_polynomial_features(X, degree):
    """
    Given X (Nx1), return polynomial features up to degree p
    """
    X_poly = X.copy()
    for d in range(2, degree + 1):
        X_poly = np.c_[X_poly, X ** d]
    return X_poly

def closed_form_linear_regression(X, y):
    X_b = np.c_[np.ones(X.shape[0]), X]  # add intercept
    beta_hat = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
    return beta_hat

def predict(X, beta):
    X_b = np.c_[np.ones(X.shape[0]), X]  # add intercept
    return X_b @ beta

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

#train polynomial models
degrees = [1, 2, 3, 4, 5]
results = []
for p in degrees:
    #create polynomial features from X
    X_train_poly = add_polynomial_features(X_train, p)
    X_test_poly = add_polynomial_features(X_test, p)
    
    #train closed-form solution
    beta_hat = closed_form_linear_regression(X_train_poly, y_train)
    
    #predictions
    y_train_pred = predict(X_train_poly, beta_hat)
    y_test_pred = predict(X_test_poly, beta_hat)
    
    #metrics
    train_mse = mse(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_mse = mse(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    results.append([p, train_mse, train_r2, test_mse, test_r2])

results_df = pd.DataFrame(results, columns=['Degree', 'Train MSE', 'Train R²', 'Test MSE', 'Test R²'])
print(results_df)

   Degree     Train MSE  Train R²       Test MSE   Test R²
0       1  57947.526161  0.496709   88575.978543  0.468736
1       2  54822.665116  0.523849   71791.679479  0.569406
2       3  53785.194716  0.532860   99833.483763  0.401216
3       4  52795.774758  0.541453  250979.274285 -0.505331
4       5  52626.111955  0.542927  570616.914821 -2.422464


The polynomial regression results demonstrate the trade-off between model complexity and generalization. For degree p=1 (linear regression), the model achieves Train R²=0.497 and Test R²=0.469, establishing a baseline performance. Increasing to p=2 (quadratic) improves both metrics, with Train R²=0.524 and Test R²=0.569, indicating that the relationship between square footage and price has some curvature that a quadratic term can capture. However, as we increase the polynomial degree further, we observe clear signs of overfitting. At p=3, the test performance begins to degrade (Test R²=0.401) despite continued improvement in training performance (Train R²=0.533). This pattern worsens dramatically for p=4 and p=5, where test R² becomes negative (-0.505 and -2.422 respectively), meaning the model performs worse than simply predicting the mean price.
The training MSE continues to decrease with higher degrees (from 57,947.53 at p=1 to 52,626.11 at p=5), but the test MSE explodes (from 88,575.98 at p=1 to 570,616.91 at p=5). This classic overfitting behavior, combined with numerical instability from unscaled features raised to high powers, shows that the model memorizes training data noise but fails to generalize. The optimal polynomial degree for this dataset is p=2, which balances model complexity with generalization ability, achieving the best test R² of 0.569.

Problem 5 

In [49]:


#features and target
cols_to_drop = [col for col in ['id', 'date', 'price'] if col in train_df.columns and col != 'price']
X_train = train_df.drop(columns=cols_to_drop + ['price']).values
y_train = train_df['price'].values.reshape(-1, 1)

cols_to_drop_test = [col for col in ['id', 'date', 'price'] if col in test_df.columns and col != 'price']
X_test = test_df.drop(columns=cols_to_drop_test + ['price']).values
y_test = test_df['price'].values.reshape(-1, 1)

#normalize features 
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test = scaler_y.transform(y_test)

#intercept column
X_train_b = np.c_[np.ones(X_train.shape[0]), X_train]
X_test_b = np.c_[np.ones(X_test.shape[0]), X_test]

#gradient descent function
def gradient_descent(X, y, alpha=0.01, iterations=100):
    m, n = X.shape
    theta = np.zeros((n, 1))
    for i in range(iterations):
        gradients = (1/m) * X.T @ (X @ theta - y)
        theta = theta - alpha * gradients
    return theta

#metrics
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

def predict(X, theta):
    return X @ theta

#gradient descent for multiple learning rates and iterations
learning_rates = [0.01, 0.1, 0.5]
iteration_checkpoints = [10, 50, 100]

results = []

for alpha in learning_rates:
    for iters in iteration_checkpoints:
        theta_hat = gradient_descent(X_train_b, y_train, alpha=alpha, iterations=iters)
        
        y_train_pred = predict(X_train_b, theta_hat)
        y_test_pred = predict(X_test_b, theta_hat)
        
        train_mse = mse(y_train, y_train_pred)
        train_r2 = r2_score(y_train, y_train_pred)
        test_mse = mse(y_test, y_test_pred)
        test_r2 = r2_score(y_test, y_test_pred)
        
        results.append([alpha, iters, theta_hat.flatten()[:5], train_mse, train_r2, test_mse, test_r2])

results_df = pd.DataFrame(results, columns=[
    "Learning Rate", "Iterations", "Theta(first 5)", 
    "Train MSE", "Train R²", "Test MSE", "Test R²"
])

print(results_df)

   Learning Rate  Iterations  \
0           0.01          10   
1           0.01          50   
2           0.01         100   
3           0.10          10   
4           0.10          50   
5           0.10         100   
6           0.50          10   
7           0.50          50   
8           0.50         100   

                                      Theta(first 5)     Train MSE  \
0  [2.050443148604586e-17, -0.0016847847544756596...  6.365384e-01   
1  [8.578749689791555e-17, -9.214032018342564e-05...  3.399825e-01   
2  [1.386683736587236e-16, 0.006322714250322141, ...  2.939048e-01   
3  [1.399596584461538e-16, 0.006563460023694356, ...  2.917823e-01   
4  [2.1834747551685585e-16, 0.023834959590630297,...  2.705807e-01   
5  [2.203857756011286e-16, 0.02554458618140569, -...  2.697378e-01   
6  [4.2587461335230614e-16, 0.7051795105315666, -...  1.117925e+04   
7  [-3.5070418859684677e-07, 562162570.3807601, -...  7.617813e+21   
8  [71788.00373578764, 7.866010103409977e+19, -1.

The gradient descent algorithm exhibits different convergence behaviors depending on the learning rate. With a small learning rate (α = 0.01), the algorithm converges slowly but steadily - after 10 iterations it achieves only Train R² = 0.36, but by 100 iterations it reaches Train R² = 0.71 and Test R² = 0.63. The optimal learning rate is α = 0.1, which balances speed and stability, achieving Train R² = 0.73 and Test R² = 0.65 after 100 iterations - nearly matching the closed-form solution (Train R² = 0.7297, Test R² = 0.6572). However, when the learning rate is too large (α = 0.5), the algorithm diverges completely. The large parameter updates cause the cost function to oscillate and explode, producing extremely large MSE values (1.49×10⁴⁴) and negative R² scores by iteration 100. This demonstrates that overshooting the optimal solution prevents convergence entirely.
Approximately 50-100 iterations are needed for appropriate learning rates to converge to a stable solution. The key finding is that feature normalization using StandardScaler was critical for convergence - without scaling features to have mean 0 and standard deviation 1, even small learning rates would cause divergence due to the vastly different scales of features like square footage (thousands) versus number of bedrooms (ones).

Problem 6

In [48]:
np.random.seed(42)  

N = 1000
X = np.random.uniform(-2, 2, N).reshape(-1, 1)  
e = np.random.normal(0, 2, N).reshape(-1, 1)    
Y = 1 + 2*X + e                                 

#intercept term
X_b = np.c_[np.ones(N), X]  

#metrics

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    return 1 - ss_res / ss_tot

#ridge closed-form solution

def ridge_closed_form(X, y, lmbda):
    n = X.shape[1]
    L = lmbda * np.eye(n)
    L[0,0] = 0  # do NOT regularize intercept
    theta = np.linalg.inv(X.T @ X + L) @ X.T @ y
    return theta


#models for different lambda
lambdas = [0, 1, 10, 100, 1000, 10000]  # 0 = normal linear regression
results = []

for lmbda in lambdas:
    theta_hat = ridge_closed_form(X_b, Y, lmbda)
    Y_pred = X_b @ theta_hat
    slope = theta_hat[1,0]
    intercept = theta_hat[0,0]
    train_mse = mse(Y, Y_pred)
    train_r2 = r2_score(Y, Y_pred)
    results.append([lmbda, intercept, slope, train_mse, train_r2])

results_df = pd.DataFrame(results, columns=['Lambda', 'Intercept', 'Slope', 'MSE', 'R²'])
print(results_df)


   Lambda  Intercept     Slope       MSE        R²
0       0   1.194775  1.922607  3.899872  0.563886
1       1   1.194720  1.921199  3.899874  0.563885
2      10   1.194230  1.908616  3.900139  0.563856
3     100   1.189658  1.791295  3.923394  0.561255
4    1000   1.163080  1.109371  4.802053  0.462997
5   10000   1.128839  0.230788  7.804391  0.127251


The ridge regression results demonstrate the effect of increasing regularization on model coefficients and performance. With λ = 0 (standard linear regression), the model estimates a slope of 1.92 and intercept of 1.19, which are close to the true values of 2.0 and 1.0 respectively, achieving MSE = 3.90 and R² = 0.564. For small regularization values (λ = 1, 10), the impact is minimal, with the slope remaining around 1.91-1.92 and performance metrics nearly unchanged, indicating that weak regularization does not significantly affect the model. As λ increases to 100, coefficient shrinkage becomes noticeable: the slope decreases to 1.79, MSE increases slightly to 3.92, and R² drops to 0.561. This shrinkage effect accelerates dramatically at higher regularization levels. At λ = 1000, the slope reduces to 1.11 (about half the true value), MSE increases to 4.80, and R² falls to 0.463. At the extreme value of λ = 10000, the model is heavily over-regularized: the slope collapses to just 0.23, MSE nearly doubles to 7.80, and R² drops to 0.127, indicating very poor performance. This progression clearly illustrates that as λ increases, ridge regression increasingly penalizes large coefficients, shrinking them toward zero. While this can prevent overfitting in complex models, for this simple linear dataset with no overfitting issues, the regularization only introduces bias and degrades model performance by pulling the coefficients away from their optimal values.